# Sparse Autoencoder Interpretability

This notebook applies Anthropic's sparse autoencoder (SAE) methodology to our speech emotion embeddings. SAEs decompose dense neural network activations into sparse, interpretable features — each feature potentially corresponding to a monosemantic concept.

**Method:** We train a simple SAE on layer-11 pooled embeddings:
- **Encoder:** Linear + ReLU (overcomplete dictionary, e.g., 2048 features from 768-dim input)
- **Decoder:** Linear reconstruction
- **Loss:** Reconstruction error + L1 sparsity penalty on the latent code

If the SAE discovers features that selectively activate for specific emotions, it provides complementary evidence (beyond linear directions) that the model's representations contain interpretable emotion-specific structure.

This directly mirrors Anthropic's approach in ["Scaling Monosemanticity"](https://transformer-circuits.pub/2024/scaling-monosemanticity/) — but applied to speech rather than language.

In [ ]:
from __future__ import annotations
import importlib.util, os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/pavannn16/speech-emotion-directions.git"
REPO_NAME = "speech-emotion-directions"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def running_in_colab():
    try: import google.colab; return True
    except ImportError: return False
IS_COLAB = running_in_colab()
print("Running in Colab:", IS_COLAB)

def run_command(cmd, cwd=None):
    print("+", " ".join(cmd)); subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)

def ensure_packages():
    pkgs = {"yaml":"pyyaml","pandas":"pandas","numpy":"numpy","matplotlib":"matplotlib",
            "seaborn":"seaborn","torch":"torch","transformers":"transformers","sklearn":"scikit-learn","tqdm":"tqdm"}
    missing = sorted({p for m,p in pkgs.items() if importlib.util.find_spec(m) is None})
    if missing: run_command([sys.executable,"-m","pip","install","-q",*missing])
    else: print("Required packages available.")

def find_local_project_root():
    cwd = Path.cwd().resolve()
    for c in [cwd, cwd.parent, cwd/"FinalProject"]:
        c = c.resolve()
        if (c/"configs"/"wav2vec.yaml").exists() and (c/"src").exists(): return c
    raise FileNotFoundError("Could not find project root.")

def clone_clean(cr):
    if cr.exists(): shutil.rmtree(cr)
    run_command(["git","clone","--depth","1",REPO_URL,str(cr)]); return cr

def prepare_roots():
    rt=Path("/content")/REPO_NAME; cl=Path("/content")/f"{REPO_NAME}-clean"
    if rt.exists() and (rt/".git").exists():
        try: run_command(["git","-C",str(rt),"fetch","origin"])
        except: pass
        st=subprocess.run(["git","-C",str(rt),"status","--short"],text=True,capture_output=True,check=True).stdout.splitlines()
        ab=subprocess.run(["git","-C",str(rt),"rev-list","--left-right","--count","HEAD...origin/main"],text=True,capture_output=True,check=False)
        a,b=(0,0) if ab.returncode!=0 or not ab.stdout.strip() else map(int,ab.stdout.strip().split())
        if not [l for l in st if l.strip()] and a==0:
            if b>0:
                try: run_command(["git","-C",str(rt),"pull","--ff-only","origin","main"]); cr=rt
                except: cr=clone_clean(cl)
            else: cr=rt
        else: cr=clone_clean(cl)
        return rt,cr
    elif rt.exists(): return rt,clone_clean(cl)
    else: run_command(["git","clone","--depth","1",REPO_URL,str(rt)]); return rt,rt

ensure_packages()
if IS_COLAB:
    from google.colab import drive; drive.mount('/content/drive',force_remount=False)
    PROJECT_ROOT,CODE_ROOT=prepare_roots()
else:
    PROJECT_ROOT=find_local_project_root(); CODE_ROOT=PROJECT_ROOT

os.chdir(PROJECT_ROOT)
for r in [str(CODE_ROOT),str(PROJECT_ROOT)]:
    while r in sys.path: sys.path.remove(r)
sys.path.insert(0,str(CODE_ROOT))
if str(PROJECT_ROOT)!=str(CODE_ROOT): sys.path.insert(1,str(PROJECT_ROOT))
for n in [n for n in sys.modules if n=="src" or n.startswith("src.")]: del sys.modules[n]
print("Project root:",PROJECT_ROOT); print("Code root:",CODE_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

experiment_name = 'wav2vec_ravdess_speaker_independent_v1'
local_analysis_dir = PROJECT_ROOT / 'artifacts' / 'analysis' / experiment_name
local_output_dir = local_analysis_dir / 'sparse_autoencoder'
local_output_dir.mkdir(parents=True, exist_ok=True)

drive_run_root = Path('/content/drive/MyDrive') / 'speech-emotion-directions' / 'runs' / experiment_name if IS_COLAB else None
drive_analysis_dir = drive_run_root / 'analysis' if drive_run_root else None
drive_output_dir = drive_analysis_dir / 'sparse_autoencoder' if drive_analysis_dir else None

analysis_source_dir = local_analysis_dir
if not (analysis_source_dir / 'embedding_arrays.npz').exists() and drive_analysis_dir and (drive_analysis_dir / 'embedding_arrays.npz').exists():
    analysis_source_dir = drive_analysis_dir

from src.analysis.emotion_vectors import load_embedding_artifacts
from src.analysis.advanced_analysis import (
    train_sparse_autoencoder_numpy, encode_with_sae,
    analyze_sae_features, sae_feature_emotion_heatmap,
)

artifacts = load_embedding_artifacts(analysis_source_dir)
label_names = list(artifacts.summary['label_names'])
label_ids = artifacts.true_label_ids
split_array = artifacts.metadata['split'].to_numpy()
train_mask = split_array == 'train'
test_mask = split_array == 'test'

final_layer_idx = int(artifacts.layer_embeddings.shape[1] - 1)
embeddings = artifacts.layer_embeddings[:, final_layer_idx]

print(f'Embedding shape: {embeddings.shape}')
print(f'Train: {train_mask.sum()}, Test: {test_mask.sum()}')

## Train Sparse Autoencoder

We use a pure-numpy SAE implementation (no PyTorch dependency for training) with an overcomplete dictionary (2048 features from 768-dim embeddings). The L1 sparsity coefficient controls how many features activate per input.

In [ ]:
train_embeddings = embeddings[train_mask]

# Normalize embeddings (zero mean, unit variance per feature)
train_mean = train_embeddings.mean(axis=0)
train_std = train_embeddings.std(axis=0).clip(1e-8)
train_normed = (train_embeddings - train_mean) / train_std

print('Training sparse autoencoder...')
sae_result = train_sparse_autoencoder_numpy(
    embeddings=train_normed,
    dictionary_size=2048,
    sparsity_coeff=5e-3,
    learning_rate=5e-4,
    num_epochs=300,
    batch_size=64,
    seed=42,
)

# Save SAE weights
np.savez_compressed(
    local_output_dir / 'sae_weights.npz',
    encoder_weight=sae_result['encoder_weight'],
    encoder_bias=sae_result['encoder_bias'],
    decoder_weight=sae_result['decoder_weight'],
    decoder_bias=sae_result['decoder_bias'],
    train_mean=train_mean,
    train_std=train_std,
)

# Plot training loss
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sae_result['loss_history'])
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (Recon + L1)')
ax.set_title('SAE Training Loss')
ax.set_yscale('log')
plt.tight_layout()
plt.savefig(local_output_dir / 'sae_training_loss.png', dpi=200)
plt.show()

print(f'Final loss: {sae_result["loss_history"][-1]:.4f}')

## Analyze SAE Activations

Encode all samples through the SAE and check:
1. How sparse are the activations?
2. Which features are most selective for each emotion?
3. Do emotion-specific features emerge?

In [ ]:
# Encode all samples
all_normed = (embeddings - train_mean) / train_std
all_activations = encode_with_sae(all_normed, sae_result['encoder_weight'], sae_result['encoder_bias'])

# Sparsity statistics
active_per_sample = (all_activations > 0).sum(axis=1)
active_features = (all_activations > 0).any(axis=0).sum()

print(f'Dictionary size: {all_activations.shape[1]}')
print(f'Active features (ever fire): {active_features} / {all_activations.shape[1]}')
print(f'Mean active features per sample: {active_per_sample.mean():.1f} ({active_per_sample.mean()/all_activations.shape[1]*100:.1f}%)')
print(f'Median active features per sample: {np.median(active_per_sample):.0f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(active_per_sample, bins=30, edgecolor='black', alpha=0.7)
ax.set_xlabel('Number of Active Features')
ax.set_ylabel('Count')
ax.set_title('SAE Sparsity: Active Features Per Sample')
ax.axvline(active_per_sample.mean(), color='red', linestyle='--', label=f'Mean: {active_per_sample.mean():.0f}')
ax.legend()
plt.tight_layout()
plt.savefig(local_output_dir / 'sae_sparsity_histogram.png', dpi=200)
plt.show()

## Most Selective Features Per Emotion

For each emotion, find the SAE features with highest selectivity ratio (class mean activation / overall mean activation). High selectivity suggests the feature is monosemantic for that emotion.

In [ ]:
feature_df = analyze_sae_features(
    activations=all_activations,
    label_ids=label_ids,
    label_names=label_names,
    top_k=10,
)
feature_df.to_csv(local_output_dir / 'sae_top_features_per_emotion.csv', index=False)

# Show top-5 for each emotion
for emo in label_names:
    print(f'\n--- {emo.upper()} ---')
    subset = feature_df[feature_df['emotion'] == emo].head(5)
    display(subset[['rank', 'feature_index', 'selectivity_ratio', 'class_mean_activation', 'activation_rate']].round(3))

## Feature-Emotion Heatmap

Visualize how the top selective features activate across emotions. A perfect monosemantic feature would show a single bright cell in one row.

In [ ]:
heatmap_matrix, selected_features, heatmap_labels = sae_feature_emotion_heatmap(
    activations=all_activations,
    label_ids=label_ids,
    label_names=label_names,
    top_k_per_class=5,
)

fig, ax = plt.subplots(figsize=(10, max(6, len(selected_features) * 0.3)))
sns.heatmap(
    heatmap_matrix,
    xticklabels=heatmap_labels,
    yticklabels=[f'feat_{i}' for i in selected_features],
    cmap='YlOrRd',
    ax=ax,
    linewidths=0.3,
)
ax.set_title('SAE Feature Activation by Emotion\n(normalized per feature, brighter = higher relative activation)')
ax.set_xlabel('Emotion')
ax.set_ylabel('SAE Feature')
plt.tight_layout()
plt.savefig(local_output_dir / 'sae_feature_emotion_heatmap.png', dpi=220)
plt.show()

## Reconstruction Quality

How well does the SAE reconstruct the original embeddings? Good reconstruction with sparse codes means the dictionary captures meaningful structure.

In [ ]:
# Reconstruction on test set
test_normed = (embeddings[test_mask] - train_mean) / train_std
test_activations = encode_with_sae(test_normed, sae_result['encoder_weight'], sae_result['encoder_bias'])
test_reconstructed = test_activations @ sae_result['decoder_weight'].T + sae_result['decoder_bias']

# Reconstruction error metrics
recon_mse = ((test_normed - test_reconstructed) ** 2).mean()
recon_cosine = np.array([
    np.dot(test_normed[i], test_reconstructed[i]) / (np.linalg.norm(test_normed[i]) * np.linalg.norm(test_reconstructed[i]) + 1e-12)
    for i in range(len(test_normed))
])

print(f'Test MSE (normalized space): {recon_mse:.4f}')
print(f'Test mean cosine similarity: {recon_cosine.mean():.4f}')
print(f'Test median cosine similarity: {np.median(recon_cosine):.4f}')

# Can we still classify from reconstructed embeddings?
from src.analysis.emotion_vectors import linear_classifier_probabilities
from sklearn.metrics import accuracy_score, f1_score
from src.analysis.extract_embeddings import load_trained_checkpoint

checkpoint_dir = PROJECT_ROOT / 'artifacts' / 'checkpoints' / experiment_name
if not (checkpoint_dir / 'model_state.pt').exists() and drive_run_root:
    checkpoint_dir = drive_run_root / 'checkpoint'

checkpoint = load_trained_checkpoint(checkpoint_dir)
cw = checkpoint.model.classifier.weight.detach().cpu().numpy()
cb = checkpoint.model.classifier.bias.detach().cpu().numpy()

# Un-normalize reconstructed embeddings and classify
test_recon_unnormed = test_reconstructed * train_std + train_mean
orig_probs = linear_classifier_probabilities(embeddings[test_mask], cw, cb)
recon_probs = linear_classifier_probabilities(test_recon_unnormed, cw, cb)

orig_preds = orig_probs.argmax(axis=1)
recon_preds = recon_probs.argmax(axis=1)
test_labels = label_ids[test_mask]

recon_metrics = pd.DataFrame([
    {'source': 'original', 'accuracy': accuracy_score(test_labels, orig_preds), 'macro_f1': f1_score(test_labels, orig_preds, average='macro')},
    {'source': 'sae_reconstructed', 'accuracy': accuracy_score(test_labels, recon_preds), 'macro_f1': f1_score(test_labels, recon_preds, average='macro')},
])
recon_metrics.to_csv(local_output_dir / 'sae_reconstruction_metrics.csv', index=False)
display(recon_metrics.round(4))

## Feature Activation Distributions

Visualize how top emotion-selective features fire across the dataset to confirm monosemanticity.

In [ ]:
# Pick top-1 most selective feature for each emotion
top_features = {}
for emo in label_names:
    subset = feature_df[feature_df['emotion'] == emo]
    if not subset.empty:
        top_features[emo] = int(subset.iloc[0]['feature_index'])

n_emos = len(top_features)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flat

all_labels = np.array([label_names[i] for i in label_ids])

for ax, (emo, feat_idx) in zip(axes, top_features.items()):
    feat_values = all_activations[:, feat_idx]
    plot_data = pd.DataFrame({'activation': feat_values, 'emotion': all_labels})
    sns.boxplot(data=plot_data, x='emotion', y='activation', ax=ax)
    ax.set_title(f'Feature {feat_idx} (top for {emo})')
    ax.set_ylabel('Activation')

# Hide extra axes if any
for i in range(len(top_features), len(list(axes))):
    axes[i].set_visible(False)

plt.suptitle('Top Selective SAE Feature Activations by Emotion', y=1.02)
plt.tight_layout()
plt.savefig(local_output_dir / 'sae_top_feature_distributions.png', dpi=220)
plt.show()

In [ ]:
# --- Drive backup ---
if not IS_COLAB:
    print('Google Drive sync is intended for Colab runtimes. Skipping.')
elif drive_output_dir is None:
    print('Drive output directory is not configured.')
else:
    import shutil
    if drive_output_dir.exists(): shutil.rmtree(drive_output_dir)
    drive_output_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_output_dir, drive_output_dir)
    print('Backed up to:', drive_output_dir)

print('\nOutput files:')
for p in sorted(local_output_dir.iterdir()):
    print(f'  {p.name}')